# Memoria del proyecto ML - Prediccion de cancelaciones hoteleras

Este documento resume el proyecto de Machine Learning realizado sobre el dataset **Hotel Booking Demand**. La memoria recoge el objetivo de negocio, la fuente de datos, la limpieza realizada, los principales hallazgos del EDA, el enfoque de modelado, la seleccion del modelo final y los proximos pasos.

La parte tecnica detallada se encuentra en los notebooks de la carpeta `notebooks` y en los scripts de `src`.


## 📖 Objetivo del proyecto

El objetivo del proyecto es construir un modelo de clasificacion capaz de predecir si una reserva hotelera sera cancelada antes de la llegada del cliente.

Desde el punto de vista de negocio, anticipar cancelaciones puede ayudar a:

- mejorar la ocupacion prevista del hotel;
- ajustar politicas de deposito o prepago;
- tomar decisiones de overbooking controlado;
- priorizar acciones comerciales sobre reservas con mayor riesgo;
- reducir perdida de ingresos por habitaciones que finalmente quedan libres.

La variable objetivo es `is_canceled`:

- `0`: reserva no cancelada;
- `1`: reserva cancelada.


## 📊 Fuente de datos

El dataset utilizado es **Hotel Booking Demand**, disponible en Kaggle.

| Archivo | Filas | Columnas |
|---|---:|---:|
| `data/raw/hotel_bookings.csv` | 119.390 | 32 |
| `data/processed/hotel_bookings_clean.csv` | 118.564 | 34 |
| `data/train/train.csv` | 94.851 | 34 |
| `data/test/test.csv` | 23.713 | 34 |

El dataset contiene informacion sobre reservas de dos tipos de hotel: `City Hotel` y `Resort Hotel`. Incluye variables sobre fechas, antelacion de reserva, noches reservadas, numero de huespedes, segmento de mercado, canal de distribucion, tipo de deposito, historial del cliente, precio medio diario y estado final de la reserva.


## 🎯 Distribucion de la target

La variable objetivo no esta perfectamente balanceada, pero tiene una proporcion suficiente de ambas clases para entrenar modelos supervisados.

| Clase | Significado | Porcentaje en processed |
|---:|---|---:|
| 0 | No cancelada | 62.74% |
| 1 | Cancelada | 37.26% |

La clase positiva del problema es `1`, es decir, la reserva cancelada.


## 📈 Limpieza y tratamiento de datos

La limpieza principal se realiza en `02_limpiezaEDA.ipynb` y queda reproducida en `src/data_processing.py`.

Decisiones principales:

- Se imputan los nulos de `children` con `0`.
- Se imputan los nulos de `country` como `Unknown`.
- Se crean variables binarias como `has_company`, `has_agent`, `has_special_requests`, `needs_parking`, `has_previous_cancellations` y `has_previous_bookings`.
- Se crean variables agregadas como `total_nights` y `total_guests`.
- Se eliminan reservas anomalas claras: reservas sin huespedes, sin noches o con `adr` negativo.
- Se eliminan variables con fuga de informacion: `reservation_status` y `reservation_status_date`.
- Se elimina `assigned_room_type` por poder representar informacion posterior a la reserva inicial.
- Se elimina `arrival_date_week_number` por ser redundante con a?o, mes y dia de llegada.
- Se eliminan `company` y `agent` como identificadores originales, pero se conserva su informacion mediante flags (`has_company`, `has_agent`).


### Decision sobre duplicados

El dataset contiene filas exactamente duplicadas. En el raw se detectan **31.994** duplicados exactos y en el processed **33.191**.

La decision final es **mantener los duplicados** por tres motivos:

- no existe una columna `booking_id` que permita confirmar que sean errores;
- dos reservas pueden compartir muchas caracteristicas aunque sean reservas distintas;
- eliminarlos cambia mucho la distribucion de la target.

Si se eliminaran duplicados en el processed, el dataset pasaria de **118.564** a **85.373** filas. Ademas, dentro de las filas duplicadas, aproximadamente el **57.40%** corresponden a reservas canceladas, por lo que eliminarlas alteraria bastante el problema de clasificacion.


## 📕 Principales conclusiones del EDA

El EDA muestra varios patrones relevantes para la cancelacion:

- Las reservas con mayor `lead_time` tienden a presentar mayor riesgo de cancelacion.
- `deposit_type` es una variable muy relevante, especialmente en reservas `Non Refund`.
- El segmento de mercado y el canal de distribucion influyen en la tasa de cancelacion.
- Las reservas sin peticiones especiales suelen cancelar mas que las que tienen peticiones especiales.
- El historial del cliente aporta informacion util: clientes con reservas anteriores no canceladas parecen tener menor riesgo.
- Variables como `adr`, `customer_type`, `market_segment`, `total_nights` y `total_guests` aportan informacion adicional para el modelo.

Estos patrones justifican usar modelos capaces de capturar relaciones no lineales e interacciones entre variables.


## 🪾 Modelos entrenados

En `03_entrenamiento_evaluacion.ipynb` se comparan cinco modelos supervisados:

1. Regresion Logistica.
2. Decision Tree.
3. Random Forest.
4. Gradient Boosting.
5. XGBoost.

Ademas, se incluye KMeans como modelo no supervisado para segmentacion exploratoria, pero no como modelo predictivo principal.

Todos los modelos supervisados usan un `Pipeline` con el mismo preprocesado:

- variables numericas: imputacion por mediana y escalado;
- variables categoricas: imputacion por moda y One-Hot Encoding;
- evaluacion con split estratificado y metrica principal `f1`.


### Seleccion del modelo final

XGBoost se selecciona como modelo final porque obtiene el mejor equilibrio entre precision, recall, F1 y ROC-AUC.

La metrica principal es `f1`, ya que en este problema interesa detectar cancelaciones reales sin generar demasiadas falsas alarmas. Tambien se revisan `precision`, `recall` y `roc_auc` para interpretar mejor el comportamiento del modelo.

La hiperparametrizacion se aborda en dos pasos:

1. `RandomizedSearchCV` para explorar una zona amplia de hiperparametros.
2. `GridSearchCV` para afinar alrededor de los mejores parametros encontrados.

El modelo final se guarda en `models/final_model.pkl` y el modelo intermedio en `models/trained_model_1.pkl`.


### Metricas del modelo final

Metricas calculadas sobre `data/test/test.csv` despues del GridSearchCV definitivo:

| Metrica | Valor |
|---|---:|
| accuracy | 0.8807 |
| precision | 0.8295 |
| recall | 0.8558 |
| f1 | 0.8425 |
| roc_auc | 0.9531 |

Matriz de confusion:

| | Prediccion no cancelada | Prediccion cancelada |
|---|---:|---:|
| Real no cancelada | 13324 | 1554 |
| Real cancelada | 1274 | 7561 |

Interpretacion rapida:

- El modelo acierta aproximadamente el 88% de las reservas del conjunto de test.
- El recall de cancelaciones es alto: detecta 7561 de las 8835 reservas canceladas reales.
- La precision tambien mejora respecto a iteraciones anteriores: cuando el modelo predice cancelacion, acierta en torno al 83% de los casos.
- El F1 final resume ese equilibrio entre precision y recall, por eso se usa como metrica principal.


### Hiperparametros principales del modelo final

| Hiperparametro | Valor |
|---|---:|
| n_estimators | 700 |
| max_depth | 10 |
| learning_rate | 0.11 |
| subsample | 0.8 |
| colsample_bytree | 0.6 |
| min_child_weight | 5 |
| gamma | 0.3 |
| reg_alpha | 1 |
| reg_lambda | 0.5 |
| scale_pos_weight | 1.68 |

Los hiperparametros definitivos salen de la busqueda fina con `GridSearchCV`. El modelo final utiliza 700 arboles, profundidad maxima 10 y un `learning_rate` de 0.11, lo que permite capturar relaciones complejas sin aumentar innecesariamente el numero de arboles.

El parametro `gamma=0.3` actua como regularizacion: exige una mejora minima para crear nuevas divisiones en los arboles. `scale_pos_weight=1.68` ayuda a compensar que hay menos reservas canceladas que no canceladas.


### Interpretabilidad

Para interpretar el modelo final se revisa la importancia de variables de XGBoost.

Como el modelo usa One-Hot Encoding, algunas variables categoricas se transforman en varias columnas internas. Por eso, en el notebook de entrenamiento se agrupan las importancias transformadas por variable original.

Esta interpretacion ayuda a entender que variables influyen mas en la prediccion y a preparar una explicacion clara para negocio.


## 🧬 Estructura del proyecto

La estructura actual queda organizada asi:

```text
Proyecto_ML/
|-- app_streamlit/              # pendiente de desarrollar
|-- data/
|   |-- raw/                    # dataset original
|   |-- processed/              # dataset limpio
|   |-- train/                  # conjunto de entrenamiento
|   `-- test/                   # conjunto de test
|-- docs/                       # documentacion y memoria
|-- models/                     # modelos pickle y configuracion
|-- notebooks/                  # flujo principal del proyecto
|-- src/                        # funciones reutilizables
`-- README.md
```

Los notebooks explican el flujo de trabajo y los scripts permiten reproducir las partes principales de limpieza, entrenamiento y evaluacion.


## 13. Limitaciones y proximos pasos

Limitaciones actuales:

- El dataset no incluye un identificador unico de reserva (`booking_id`).
- La decision sobre duplicados se justifica, pero no se puede validar al 100% sin informacion adicional.
- El modelo se evalua con split train/test estratificado, no con una validacion temporal estricta.
- Algunas variables pueden depender del momento exacto en que se quiera hacer la prediccion en un caso real.

Proximos pasos:

- Pulir la demo de Streamlit para la presentacion final y conectarla al relato de negocio.
- Preparar una presentacion orientada a negocio con demo.
- Revisar si conviene ajustar el threshold segun el coste de falsos positivos y falsos negativos.
- Documentar claramente como se usaria el modelo en un flujo real de reservas hoteleras.


## 14. Conclusiones

El proyecto consigue construir un flujo completo de Machine Learning para predecir cancelaciones hoteleras:

- se parte de un dataset real con volumen suficiente;
- se realiza limpieza, EDA y feature engineering;
- se comparan cinco modelos supervisados;
- se justifica mantener duplicados;
- se selecciona XGBoost como modelo final;
- se mejora XGBoost mediante `RandomizedSearchCV` y un `GridSearchCV` definitivo;
- se obtiene un modelo final con `f1=0.8425` y `roc_auc=0.9531` sobre test;
- se guarda el modelo final, un modelo intermedio y su configuracion;
- se crea una primera demo de Streamlit para realizar predicciones interactivas.

El resultado es un modelo con buen rendimiento predictivo y una interpretacion clara desde el punto de vista de negocio: permite anticipar cancelaciones con suficiente precision para apoyar decisiones de ocupacion, ingresos y overbooking controlado.


In [ ]:
# Celda opcional para revisar rapidamente los archivos principales del proyecto
from pathlib import Path

candidates = [Path.cwd(), Path.cwd().parent]
ROOT = next(path for path in candidates if (path / "data").exists() and (path / "models").exists())

for path in [
    ROOT / "data" / "raw" / "hotel_bookings.csv",
    ROOT / "data" / "processed" / "hotel_bookings_clean.csv",
    ROOT / "data" / "train" / "train.csv",
    ROOT / "data" / "test" / "test.csv",
    ROOT / "models" / "trained_model_1.pkl",
    ROOT / "models" / "final_model.pkl",
    ROOT / "models" / "model_config.yaml",
]:
    print(path, "OK" if path.exists() else "NO EXISTE")
